In [38]:
import numpy as np
import joblib
from scipy.special import logsumexp

# Load the saved model
obj = joblib.load("hmm_model.pkl")

# If the file stores the mappings too
if isinstance(obj, dict):
    model = obj.get("model", obj)
    char_to_int = obj.get("char_to_int")
    int_to_char = obj.get("int_to_char")
else:
    raise RuntimeError("Mapping (char_to_int) not found in saved model. Please include it when saving.")

# Extract key parameters from trained model
N = model.n_components                     # Number of hidden states
V = model.emissionprob_.shape[1]           # Vocabulary size
log_start = np.log(model.startprob_ + 1e-12)
log_trans = np.log(model.transmat_ + 1e-12)
log_emit  = np.log(model.emissionprob_ + 1e-12)


In [39]:
# ----------------------------
# Convert Masked Pattern to Indexed Form
# ----------------------------
def pattern_to_indexed(masked_pattern):
    """
    Converts '_a_e' → [None, index('a'), None, index('e')]
    None represents unknown letters (blanks).
    """
    seq = []
    for ch in masked_pattern:
        if ch == '_' or ch == '?':
            seq.append(None)
        else:
            seq.append(char_to_int[ch])
    return seq


In [40]:
# ----------------------------
# Forward-Backward Algorithm (in log space)
# ----------------------------
def compute_gamma_log(masked_idx_seq):
    """
    Compute log posterior probability of each hidden state
    at each word position using forward-backward algorithm.
    """
    T = len(masked_idx_seq)
    alpha = np.full((T, N), -np.inf)
    beta  = np.full((T, N), -np.inf)

    # --- Forward pass ---
    if masked_idx_seq[0] is None:
        alpha[0] = log_start
    else:
        o = masked_idx_seq[0]
        alpha[0] = log_start + log_emit[:, o]

    for t in range(1, T):
        o = masked_idx_seq[t]
        emit_log = 0.0 if o is None else log_emit[:, o]
        alpha[t] = emit_log + logsumexp(alpha[t-1][:, None] + log_trans, axis=0)

    # --- Backward pass ---
    beta[T-1] = 0.0
    for t in range(T-2, -1, -1):
        next_o = masked_idx_seq[t+1]
        emit_next = 0.0 if next_o is None else log_emit[:, next_o]
        beta[t] = logsumexp(log_trans + (emit_next + beta[t+1])[None, :], axis=1)

    # Combine forward & backward
    gamma_log = alpha + beta
    gamma_log = gamma_log - logsumexp(gamma_log, axis=1, keepdims=True)
    return gamma_log


In [41]:
# ----------------------------
# Compute Letter Probabilities (Marginals)
# ----------------------------
def get_position_letter_marginals(masked_pattern):
    masked_idx = pattern_to_indexed(masked_pattern)
    gamma_log = compute_gamma_log(masked_idx)
    gamma = np.exp(gamma_log)

    marginals = []
    for t, obs in enumerate(masked_idx):
        if obs is None:
            probs = gamma[t] @ model.emissionprob_
            probs = probs / (probs.sum() + 1e-12)
            marginals.append(probs)
        else:
            onehot = np.zeros(V)
            onehot[obs] = 1.0
            marginals.append(onehot)
    return marginals


In [42]:
# ----------------------------
# Predict Next Letter Probabilities
# ----------------------------
letter_indexes = [char_to_int[c] for c in list("abcdefghijklmnopqrstuvwxyz")]

def get_letter_probs_for_mask(masked_pattern, guessed_set):
    marginals = get_position_letter_marginals(masked_pattern)
    blanks = [i for i, ch in enumerate(masked_pattern) if ch in ['_', '?']]
    if not blanks:
        return np.zeros(26)
    pos = blanks[0]
    probs_v = marginals[pos]
    probs_26 = np.array([probs_v[idx] for idx in letter_indexes])

    # remove already guessed letters
    for i, ch in enumerate(list("abcdefghijklmnopqrstuvwxyz")):
        if ch in guessed_set:
            probs_26[i] = 0.0
    s = probs_26.sum()
    probs_26 = probs_26 / s if s > 0 else np.ones(26) / 26
    return probs_26


In [43]:
# ----------------------------
# DEMO: Check model predictions
# ----------------------------
patterns = ["_a_e", "m_ch_ne", "ba_ana", "pyth_n", "_a_a_a"]
guessed = {'a', 'e', 't', 'o', 'n'}  # example already-guessed letters

for pattern in patterns:
    probs = get_letter_probs_for_mask(pattern, guessed)
    letters = list("abcdefghijklmnopqrstuvwxyz")
    top5 = sorted(zip(letters, probs), key=lambda x: x[1], reverse=True)[:5]
    print(f"\nPattern: {pattern}")
    for l, p in top5:
        print(f"  {l}: {p:.3f}")



Pattern: _a_e
  r: 0.148
  l: 0.135
  h: 0.117
  u: 0.081
  c: 0.072

Pattern: m_ch_ne
  i: 0.462
  u: 0.179
  y: 0.064
  s: 0.055
  p: 0.049

Pattern: ba_ana
  r: 0.168
  l: 0.126
  m: 0.112
  d: 0.079
  c: 0.073

Pattern: pyth_n
  i: 0.667
  u: 0.181
  y: 0.107
  r: 0.007
  q: 0.007

Pattern: _a_a_a
  r: 0.154
  h: 0.113
  l: 0.112
  u: 0.102
  c: 0.069
